# Capability Analysis

This notebook contains code for extracting all the capability information
of the capability files and performing analyses on that data.

In [1]:
import json
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

In [2]:
# pd.set_option('display.max_rows', None)

## Preparation

Extract all the JSON arrays of all `capabilities.json` files in the directory. The `capabilitiers.json`

**One capability per image per row.** The goal is to create **Tidy Data**, thus we **normalize** it.

In [3]:
def load_caps_from_ipsw_dir(path):
    """Load capabilities.json JSON file and parse it"""
    with open(f"{path}/capabilities.json") as f:
        return json.load(f)

def parse_filename(filename):
    """Extract information about device name and OS version of the path name"""
    pattern = r"^(?P<devices>.+)_(?P<os_version>\d+(?:\.\d+)*)_(?P<build>[0-9A-Za-z]+)_Restore\.ipsw$"
    m = re.match(pattern, filename)
    if not m:
        raise ValueError(f"Unrecognized file format: {filename}")
    return m.groups()

DEVICE_ID_RE = re.compile(r"[A-Za-z]+[0-9]+,\d+")

def split_devices(devices_raw: str) -> list[str]:
    ids = DEVICE_ID_RE.findall(devices_raw)
    if ids and "," in devices_raw:
        return ids
    return [devices_raw]

In [4]:
rows = []

ipsw_dir = Path("capabilities-ipsw/capabilities-results/")

for ipsw in ipsw_dir.glob("*.ipsw"):
    device_model_raw, os_version, build = parse_filename(ipsw.name)
    device_models = split_devices(device_model_raw)

    caps = load_caps_from_ipsw_dir(ipsw)

    for cap, value in caps.items():
        rows.append({
            "device_models": device_models,
            "os_version": os_version,
            "build": build,
            "name": cap,
            "raw_value": value,
        })

df = pd.DataFrame(rows)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11811 entries, 0 to 11810
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   device_models  11811 non-null  object
 1   os_version     11811 non-null  object
 2   build          11811 non-null  object
 3   name           11811 non-null  object
 4   raw_value      11807 non-null  object
dtypes: object(5)
memory usage: 461.5+ KB


In [6]:
df.describe()

,device_models,os_version,build,name,raw_value
count,11811,11811,11811,11811,11807
unique,53,50,64,123,5
top,"[iPhone14,4]",26.3,23D5089e,show-peer-errors,available
freq,2564,5755,5400,123,10872


Some images relate to multiple devices. We normalize them as well, so every capability entry is recorded for each device.

In [7]:
df = df.explode("device_models", ignore_index=True).rename(columns={"device_models": "device_model"})

In [8]:
to_float_mask = (df["name"] == "ec-version") | (df["name"] == "kt-version")
df.loc[to_float_mask, "raw_value"] = df.loc[to_float_mask, "raw_value"].apply(lambda x: float(x) if isinstance(x, (int, float)) else x)

In [9]:
df[to_float_mask]

,device_model,os_version,build,name,raw_value
28,"iPhone14,4",15.5,19F77,kt-version,4.0
30,"iPhone14,4",15.5,19F77,ec-version,1.0
113,"iPhone12,3",26.3,23D5089e,kt-version,5.0
114,"iPhone12,5",26.3,23D5089e,kt-version,5.0
117,"iPhone12,3",26.3,23D5089e,ec-version,1.0
...,...,...,...,...,...
13107,UniversalMac,15.1,24B83,ec-version,1.0
13196,"iPhone18,3",26.3,23D5089e,kt-version,5.0
13198,"iPhone18,3",26.3,23D5089e,ec-version,1.0
13316,"iPhone14,7",26.3,23D5089e,kt-version,5.0


Extracted capabilities from the JSON can contain the following values:

* `"available"` denotes that a symbol for that capability could be found in that image. The value was not reversed yet, thus the capability could be of any value or not sent at all.
* "literal" values like
  * booleans: `True` or `False`
  * strings
  * floats
  * ints

In order to distinguish between

* a capability we that exists in that version and
* a capability we know the exact value,

we create classify it.

In [10]:
# if value is "available", it's not a String, it's just a marker
# for it that a symbol is just present
AVAIL_SENTINEL = "available"

# Grab the source column
s = df["raw_value"]

# Identitfy the rows that are exactly the sentinel string.
is_just_available = s.eq(AVAIL_SENTINEL) & s.map(type).eq(str)

df["value_kind"] = pd.Categorical(
    is_just_available.map({True: "available", False: "literal"}),
    categories=["available", "literal"]
)

# Replace the sentinel with NA so "available" rows have no typed value.
# Literal rows keep their orignal value (str, int, float, ...)
df["typed_value"] = s.where(~is_just_available, pd.NA)

df["value_type"] = pd.Series(pd.NA, index=df.index, dtype="string")
lit = ~is_just_available
df.loc[lit, "value_type"] = s.loc[lit].map(type).map(lambda t: t.__name__)

df["present"] = 1 # for pivot operations

Examples for capabilities with capabilities that are just available:

In [11]:
df[df["value_kind"].eq("available")].head()

,device_model,os_version,build,name,raw_value,value_kind,typed_value,value_type,present
0,"iPhone14,4",15.5,19F77,show-peer-errors,available,available,<NA>,<NA>,1
1,"iPhone14,4",15.5,19F77,is-c2k-equipment,available,available,<NA>,<NA>,1
2,"iPhone14,4",15.5,19F77,supports-inline-attachments,available,available,<NA>,<NA>,1
3,"iPhone14,4",15.5,19F77,supports-keep-receipts,available,available,<NA>,<NA>,1
4,"iPhone14,4",15.5,19F77,supports-location-sharing,available,available,<NA>,<NA>,1


Examples for capabilities with capabilities that we have literal values for:

In [12]:
df[df["value_kind"].eq("literal")].head()

,device_model,os_version,build,name,raw_value,value_kind,typed_value,value_type,present
28,"iPhone14,4",15.5,19F77,kt-version,4.0,literal,4.0,float,1
30,"iPhone14,4",15.5,19F77,ec-version,1.0,literal,1.0,float,1
32,"iPhone14,4",15.5,19F77,supports-heif,True,literal,True,bool,1
43,"iPhone14,4",15.5,19F77,supports-activity-sharing-v1,True,literal,True,bool,1
47,"iPhone14,4",15.5,19F77,supports-fm-fence-v1,True,literal,True,bool,1


Each capability can identified by the key: `(device_model, os_version, name)`

In [13]:
df["device_model"].value_counts().index

Index(['iPhone14,4', 'UniversalMac', 'iPad_Pro_Spring_2021', 'iPhone17,3',
       'Apple_Vision_Pro', 'iPhone14,7', 'iPad_Spring_2022', 'iPhone15,2',
       'iPad_10.2', 'iPhone10,6', 'iPhone10,3', 'iPhone12,3', 'iPad_Fall_2021',
       'iPad_Fall_2020', 'iPhone14,3', 'iPhone16,2', 'iPhone17,5',
       'iPhone13,4', 'iPad15,4', 'iPhone15,5', 'iPad_Fall_2022', 'iPhone14,8',
       'iPhone17,4', 'iPhone15,3', 'iPad15,3', 'iPhone12,8', 'iPhone18,1',
       'iPad15,5', 'iPad15,6', 'iPhone12,1', 'iPhone15,4', 'iPhone17,1',
       'iPhone12,5', 'iPad_Spring_2019', 'iPad14,6', 'iPhone14,2',
       'iPhone14,5', 'iPad_Pro_M4', 'iPad_10.2_2020', 'iPad_Air_M2',
       'iPhone13,2', 'iPhone13,3', 'iPad17,3', 'iPad15,8',
       'iPad_Pro_A12X_A12Z', 'iPad17,1', 'iPad17,4', 'iPad17,2', 'iPhone16,1',
       'iPhone18,2', 'iPhone17,2', 'iPhone13,1', 'iPhone18,4', 'iPad15,7',
       'iPhone14,6', 'iPad14,3', 'iPad14,4', 'iPad14,5', 'iPhone18,3',
       'iPad_10.2_2021', 'iPad16,1', 'iPad16,2', 'Apple_

## Cleanup

In [14]:
remove_incomplete = False
if remove_incomplete:
    df = df[~df["device_model"].str.contains("Watch|Vision", na=False)]

### Deduplication

Find and remove all rows with duplicate values.

In [15]:
dedup_cols = ["device_model", "os_version", "name"]

Find possible conflicts with literal values that have to be resolved in advance.

In [16]:
(df[df["value_kind"].eq("literal")]
    .groupby(dedup_cols)["typed_value"]
    .nunique(dropna=False)
    .reset_index(name="distinct_literal_values")
    .query("distinct_literal_values > 1"))

,device_model,os_version,name,distinct_literal_values


Remove duplicates

In [17]:
df = df.drop_duplicates(dedup_cols).copy()

## Extract extra information

In [18]:
def parse_os_version(v):
    """
    '26.2' -> (26, 2)
    """
    return tuple(int(p) for p in v.split("."))

df["os_version_key"] = df["os_version"].map(parse_os_version)

In [19]:
def device_type_from_model(model: str):
    if model.startswith("iPhone"):
        return "iPhone"
    if model.startswith("iPad"):
        return "iPad"
    if model.startswith("Watch"):
        return "Watch"
    if "Mac" in model:
        return "Mac"
    if "Vision" in model:
        return "VisionPro"
    raise ValueError(f"Unknown Device Type from model: {model}")

df["device_type"] = df["device_model"].map(device_type_from_model)

We now have the prepared DataFrame.

In [20]:
df

,device_model,os_version,build,name,raw_value,value_kind,typed_value,value_type,present,os_version_key,device_type
0,"iPhone14,4",15.5,19F77,show-peer-errors,available,available,<NA>,<NA>,1,"(15, 5)",iPhone
1,"iPhone14,4",15.5,19F77,is-c2k-equipment,available,available,<NA>,<NA>,1,"(15, 5)",iPhone
2,"iPhone14,4",15.5,19F77,supports-inline-attachments,available,available,<NA>,<NA>,1,"(15, 5)",iPhone
3,"iPhone14,4",15.5,19F77,supports-keep-receipts,available,available,<NA>,<NA>,1,"(15, 5)",iPhone
4,"iPhone14,4",15.5,19F77,supports-location-sharing,available,available,<NA>,<NA>,1,"(15, 5)",iPhone
...,...,...,...,...,...,...,...,...,...,...,...
13403,"iPhone14,7",26.3,23D5089e,supports-legacy-contact-invites-v2,available,available,<NA>,<NA>,1,"(26, 3)",iPhone
13404,"iPhone14,7",26.3,23D5089e,supports-legacy-contact-invites-v3,available,available,<NA>,<NA>,1,"(26, 3)",iPhone
13405,"iPhone14,7",26.3,23D5089e,supports-calls-25,available,available,<NA>,<NA>,1,"(26, 3)",iPhone
13406,"iPhone14,7",26.3,23D5089e,supports-introductions-v1,available,available,<NA>,<NA>,1,"(26, 3)",iPhone


## Hardware information

In [21]:
def set_value_for_row(df, mask, v):
    df.loc[mask, "value_kind"] = "literal"
    df.loc[mask, "typed_value"] = v
    df.loc[mask, "value_type"] = type(v).__name__

In [22]:
def get_device_cap_mask(df, device_regex, cap_name, inverse_dev=False):
    devices = df["device_model"].str.match(device_regex, na=False)
    caps = (df["name"] == cap_name)
    if inverse_dev:
        return (~devices) & caps
    return devices & caps

In [23]:
stewie_devices = get_device_cap_mask(df, r"^iPhone(1[4-9]),", "supports-stewie")
set_value_for_row(df, stewie_devices, True)
df[stewie_devices].size

418

In [24]:
non_stewie_devices = get_device_cap_mask(df, r"^iPhone(1[4-9]),", "supports-stewie", inverse_dev=True)
set_value_for_row(df, non_stewie_devices, False)
df[non_stewie_devices].head(5)

,device_model,os_version,build,name,raw_value,value_kind,typed_value,value_type,present,os_version_key,device_type
269,"iPhone12,3",26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPhone
270,"iPhone12,5",26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPhone
460,iPad_Spring_2019,26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPad
820,iPad_Fall_2020,26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPad
1016,iPad_Pro_Spring_2021,26.0,23A341,supports-stewie,available,literal,False,bool,1,"(26, 0)",iPad


In [25]:
df[df["name"] == "supports-stewie"].size

1012

## Statistics for Paper

In [26]:
df["device_model"].value_counts()

device_model
iPhone14,4                 2564
UniversalMac               1352
iPad_Pro_Spring_2021        714
iPhone17,3                  476
Apple_Vision_Pro            411
                           ... 
iPad16,2                    120
Apple_Vision_Pro_M5         119
iPad_64bit_TouchID_ASTC      70
iPhone_4.7_P3                69
iPad_Educational             40
Name: count, Length: 66, dtype: int64

In [27]:
df["build"].value_counts()

build
23D5089e    6840
22E240       396
23C55        240
23N5588c     238
23A341       236
            ... 
19G71         57
21A559        50
20B29         37
17A844        31
15A402        19
Name: count, Length: 64, dtype: int64

In [28]:
df["device_type"].value_counts()

device_type
iPhone       7142
iPad         4264
Mac          1352
VisionPro     530
Name: count, dtype: int64

In [29]:
df.groupby(["device_model", "os_version_key"])["name"].count().sort_values()#.to_csv("deivce_os_version_cap_count.csv")#.agg(_min="min", _max="max")

device_model             os_version_key
iPad_64bit_TouchID_ASTC  (11, 0, 1)         19
                         (12, 0)            20
iPhone10,3               (12, 0)            20
iPhone10,6               (12, 0)            20
                         (13, 0)            30
                                          ... 
iPhone18,1               (26, 3)           120
iPhone18,2               (26, 3)           120
iPhone17,4               (26, 3)           120
iPhone18,3               (26, 3)           120
iPhone18,4               (26, 3)           120
Name: name, Length: 138, dtype: int64

In [30]:
df.groupby(["device_model", "os_version_key"])["name"].count().reset_index().sort_values(["os_version_key"]).to_csv("os_count.csv")

In [31]:
df["typed_value"].value_counts()

typed_value
1.0      812
False    169
5.0      113
4.0       18
Name: count, dtype: int64

In [32]:
df[df["typed_value"] == False]

,device_model,os_version,build,name,raw_value,value_kind,typed_value,value_type,present,os_version_key,device_type
269,"iPhone12,3",26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPhone
270,"iPhone12,5",26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPhone
426,iPad_Spring_2019,26.3,23D5089e,supports-uwb,False,literal,False,bool,1,"(26, 3)",iPad
460,iPad_Spring_2019,26.3,23D5089e,supports-stewie,available,literal,False,bool,1,"(26, 3)",iPad
786,iPad_Fall_2020,26.3,23D5089e,supports-uwb,False,literal,False,bool,1,"(26, 3)",iPad
...,...,...,...,...,...,...,...,...,...,...,...
12936,Apple_Vision_Pro,1.0.2,21N323,supports-uwb,False,literal,False,bool,1,"(1, 0, 2)",VisionPro
13078,UniversalMac,15.1,24B83,is-c2k-equipment,False,literal,False,bool,1,"(15, 1)",Mac
13127,UniversalMac,15.1,24B83,supports-harmony,False,literal,False,bool,1,"(15, 1)",Mac
13135,UniversalMac,15.1,24B83,supports-uwb,False,literal,False,bool,1,"(15, 1)",Mac


In [33]:
df.shape

(13288, 11)

## Analysis: OS version evolution

In [34]:
cap_matrix = (
    df.pivot_table(
        index=["device_model", "os_version"],
        columns="name",
        values="present",
        fill_value=0,
        aggfunc="max",
    )
    .sort_index()
)

cap_matrix

name                            device-key-signature  ec-version  \
device_model        os_version                                     
Apple_Vision_Pro    1.0.2                          1           1   
                    2.0                            1           1   
                    26.0                           1           1   
                    26.3                           1           1   
Apple_Vision_Pro_M5 26.3                           1           1   
...                                              ...         ...   
iPhone18,1          26.3                           1           1   
iPhone18,2          26.3                           1           1   
iPhone18,3          26.3                           1           1   
iPhone18,4          26.3                           1           1   
iPhone_4.7_P3       16.7.12                        1           1   

name                            is-c2k-equipment  is-green-tea  is-web-client  \
device_model        os_version                                                  
Apple_Vision_Pro    1.0.2                      1             1              1   
                    2.0                        1             1              1   
                    26.0                       1             1              1   
                    26.3                       1             1              1   
Apple_Vision_Pro_M5 26.3                       1             1              1   
...                                          ...           ...            ...   
iPhone18,1          26.3                       1             1              1   
iPhone18,2          26.3                       1             1              1   
iPhone18,3          26.3                       1             1              1   
iPhone18,4          26.3                       1             1              1   
iPhone_4.7_P3       16.7.12                    1             1              1   

name                            kt-version  nicknames-version  \
device_model        os_version                                  
Apple_Vision_Pro    1.0.2                1                  1   
                    2.0                  1                  1   
                    26.0                 1                  1   
                    26.3                 1                  1   
Apple_Vision_Pro_M5 26.3                 1                  1   
...                                    ...                ...   
iPhone18,1          26.3                 1                  1   
iPhone18,2          26.3                 1                  1   
iPhone18,3          26.3                 1                  1   
iPhone18,4          26.3                 1                  1   
iPhone_4.7_P3       16.7.12              1                  1   

name                            optionally-receive-typing-indicators  \
device_model        os_version                                         
Apple_Vision_Pro    1.0.2                                          1   
                    2.0                                            1   
                    26.0                                           1   
                    26.3                                           1   
Apple_Vision_Pro_M5 26.3                                           1   
...                                                              ...   
iPhone18,1          26.3                                           1   
iPhone18,2          26.3                                           1   
iPhone18,3          26.3                                           1   
iPhone18,4          26.3                                           1   
iPhone_4.7_P3       16.7.12                                        1   

name                            prefers-sdr  sender-key-message-version  ...  \
device_model        os_version                                           ...   
Apple_Vision_Pro    1.0.2                 1                           0  ...   
                    2.0               

In [35]:
cap_counts = (
    cap_matrix
        .sum(axis=1)
        .rename("num_capabilities")
        .reset_index()
)

cap_counts["os_version_key"] = cap_counts["os_version"].map(parse_os_version)

cap_counts = cap_counts.sort_values("os_version_key")

cap_counts = cap_counts.merge(
    df[["device_model", "device_type"]].drop_duplicates(),
    on="device_model"
)

cap_counts

,device_model,os_version,num_capabilities,os_version_key,device_type
0,Apple_Vision_Pro,1.0.2,81,"(1, 0, 2)",VisionPro
1,Apple_Vision_Pro,2.0,94,"(2, 0)",VisionPro
2,UniversalMac,11.0.1,37,"(11, 0, 1)",Mac
3,iPad_64bit_TouchID_ASTC,11.0.1,19,"(11, 0, 1)",iPad
4,"iPhone10,6",12.0,20,"(12, 0)",iPhone
...,...,...,...,...,...
133,"iPad16,2",26.3,120,"(26, 3)",iPad
134,"iPhone13,2",26.3,120,"(26, 3)",iPhone
135,"iPhone13,1",26.3,120,"(26, 3)",iPhone
136,"iPad15,8",26.3,120,"(26, 3)",iPad


In [36]:
cap_num_hist = cap_counts.groupby(["device_type", "os_version"])["num_capabilities"].agg(["min", "median", "max"])
cap_num_hist

min  median  max
device_type os_version                  
Mac         11.0.1       37    37.0   37
            12.0.1       50    50.0   50
            13.0         64    64.0   64
            14.0         78    78.0   78
            14.3         78    78.0   78
...                     ...     ...  ...
iPhone      26.0        118   118.0  118
            26.0.1      118   118.0  118
            26.1        120   120.0  120
            26.2        120   120.0  120
            26.3        120   120.0  120

[72 rows x 3 columns]

In [37]:
cnh = cap_num_hist.loc["iPhone"]["max"].reset_index()
cnh["version_key"] = cnh["os_version"].map(parse_os_version)
cnh["major"] = cnh["version_key"].map(lambda s: s[0])
print(cnh.sort_values("version_key").groupby("major").head(1).set_index("major"))

# .plot(kind="bar")

#plt.xlabel("iOS Major Version")
#plt.ylabel("Number of Capabilities")
#plt.legend().remove()
#plt.savefig("iOS_Caps_Major_Version.pdf")
#plt.show()

      os_version  max version_key
major                            
12          12.0   20     (12, 0)
13          13.0   30     (13, 0)
14          14.0   40     (14, 0)
15          15.0   53     (15, 0)
16          16.0   67     (16, 0)
17          17.0   81     (17, 0)
18          18.0   94     (18, 0)
26          26.0  118     (26, 0)


In [38]:
cnh = cap_num_hist.loc["Mac"]["max"].reset_index()
cnh["version_key"] = cnh["os_version"].map(parse_os_version)
cnh["major"] = cnh["version_key"].map(lambda s: s[0])
print(cnh.sort_values("version_key").groupby("major").head(1).set_index("major"))#.plot(kind="bar")

#plt.xlabel("iOS Major Version")
#plt.ylabel("Number of Capabilities")
#plt.legend().remove()
#plt.savefig("macOS_Caps_Major_Version.pdf")
#plt.show()

      os_version  max version_key
major                            
11        11.0.1   37  (11, 0, 1)
12        12.0.1   50  (12, 0, 1)
13          13.0   64     (13, 0)
14          14.0   78     (14, 0)
15          15.0   91     (15, 0)
26          26.0  115     (26, 0)


In [39]:
cnh = cap_num_hist.loc["iPad"]["max"].reset_index()
cnh["version_key"] = cnh["os_version"].map(parse_os_version)
cnh["major"] = cnh["version_key"].map(lambda s: s[0])
print(cnh.sort_values("version_key").groupby("major").head(1).set_index("major"))#.plot(kind="bar")

#plt.xlabel("iOS Major Version")
#plt.ylabel("Number of Capabilities")
#plt.legend().remove()
#plt.savefig("iPadOS_Caps_Major_Version.pdf")
#plt.show()

      os_version  max version_key
major                            
11        11.0.1   19  (11, 0, 1)
12          12.0   20     (12, 0)
13          13.1   31     (13, 1)
14          14.0   40     (14, 0)
15          15.0   53     (15, 0)
16          16.1   67     (16, 1)
17          17.0   81     (17, 0)
18          18.0   94     (18, 0)
26          26.0  118     (26, 0)


In [40]:
first_last_seen = (
    df
        .sort_values(["device_type", "name", "os_version_key"], kind="mergesort")
        .groupby(["device_type", "name"], as_index=False)
        # .first()[["device_type", "name", "os_version", "os_version_key"]]
        # .rename(columns={
        #     "os_version": "first_os_version",
        #     "os_version_key": "first_os_version_key",
        # })
        .agg(
            first_os_version=("os_version", "first"),
            first_os_version_key=("os_version_key", "first"),
            last_os_version=("os_version", "last"),
            last_os_version_key=("os_version_key", "last"),
        )
)

In [41]:
first_last_seen

,device_type,name,first_os_version,first_os_version_key,last_os_version,last_os_version_key
0,Mac,device-key-signature,13.0,"(13, 0)",26.3,"(26, 3)"
1,Mac,ec-version,11.0.1,"(11, 0, 1)",26.3,"(26, 3)"
2,Mac,is-c2k-equipment,11.0.1,"(11, 0, 1)",26.3,"(26, 3)"
3,Mac,is-green-tea,12.0.1,"(12, 0, 1)",26.3,"(26, 3)"
4,Mac,is-web-client,13.0,"(13, 0)",26.3,"(26, 3)"
...,...,...,...,...,...,...
477,iPhone,supports-update-attachments-v1,12.0,"(12, 0)",26.3,"(26, 3)"
478,iPhone,supports-uriless-membership-updates,26.0,"(26, 0)",26.3,"(26, 3)"
479,iPhone,supports-user-driven-call-activation,18.4,"(18, 4)",26.3,"(26, 3)"
480,iPhone,supports-uwb,15.0,"(15, 0)",26.3,"(26, 3)"


In [43]:
first_last_seen[first_last_seen["last_os_version_key"] < first_last_seen["last_os_version_key"].max()]

,device_type,name,first_os_version,first_os_version_key,last_os_version,last_os_version_key
88,Mac,supports-queue-one-read-receipts,26.0,"(26, 0)",26.0,"(26, 0)"
207,VisionPro,supports-queue-one-read-receipts,26.0,"(26, 0)",26.0,"(26, 0)"
326,iPad,supports-protobuf-payload-data-v1,13.1,"(13, 1)",13.1,"(13, 1)"
329,iPad,supports-queue-one-read-receipts,26.0,"(26, 0)",26.0,"(26, 0)"
450,iPhone,supports-queue-one-read-receipts,26.0,"(26, 0)",26.1,"(26, 1)"
451,iPhone,supports-queue-one-read-receipts-v2,26.1,"(26, 1)",26.1,"(26, 1)"


In [44]:
first_last_seen[first_last_seen["name"].str.startswith("supports-askto")]

,device_type,name,first_os_version,first_os_version_key,last_os_version,last_os_version_key
15,Mac,supports-askto,14.0,"(14, 0)",26.3,"(26, 3)"
16,Mac,supports-askto-responseUI,26.3,"(26, 3)",26.3,"(26, 3)"
17,Mac,supports-askto-v2,26.0,"(26, 0)",26.3,"(26, 3)"
133,VisionPro,supports-askto,1.0.2,"(1, 0, 2)",26.3,"(26, 3)"
134,VisionPro,supports-askto-responseUI,26.3,"(26, 3)",26.3,"(26, 3)"
135,VisionPro,supports-askto-v2,26.0,"(26, 0)",26.3,"(26, 3)"
254,iPad,supports-askto,17.0,"(17, 0)",26.3,"(26, 3)"
255,iPad,supports-askto-responseUI,26.2,"(26, 2)",26.3,"(26, 3)"
256,iPad,supports-askto-v2,26.0,"(26, 0)",26.3,"(26, 3)"
376,iPhone,supports-askto,17.0,"(17, 0)",26.3,"(26, 3)"


In [45]:
first_last_seen[first_last_seen["name"].str.contains("heif")]

,device_type,name,first_os_version,first_os_version_key,last_os_version,last_os_version_key
50,Mac,supports-heif,11.0.1,"(11, 0, 1)",26.3,"(26, 3)"
168,VisionPro,supports-heif,1.0.2,"(1, 0, 2)",26.3,"(26, 3)"
289,iPad,supports-heif,14.0,"(14, 0)",26.3,"(26, 3)"
411,iPhone,supports-heif,14.0,"(14, 0)",26.3,"(26, 3)"


In [46]:
first_last_seen[(first_last_seen["first_os_version_key"] >= (18,0)) & (first_last_seen["first_os_version_key"] < (19,0))]

,device_type,name,first_os_version,first_os_version_key,last_os_version,last_os_version_key
247,iPad,sender-key-message-version,18.0,"(18, 0)",26.3,"(26, 3)"
271,iPad,supports-emoji-images,18.0,"(18, 0)",26.3,"(26, 3)"
273,iPad,supports-emoji-tapbacks,18.0,"(18, 0)",26.3,"(26, 3)"
291,iPad,supports-high-quality-photo-file-sizes,18.0,"(18, 0)",26.3,"(26, 3)"
301,iPad,supports-legacy-contact-invites-v1,18.4,"(18, 4)",26.3,"(26, 3)"
308,iPad,supports-manatee-activity-sharing,18.0,"(18, 0)",26.3,"(26, 3)"
328,iPad,supports-qta,18.7.1,"(18, 7, 1)",26.3,"(26, 3)"
331,iPad,supports-rbm-chatbot,18.4,"(18, 4)",26.3,"(26, 3)"
332,iPad,supports-recovery-contact-upsell,18.4,"(18, 4)",26.3,"(26, 3)"
336,iPad,supports-remote-atv-signin,18.0,"(18, 0)",26.3,"(26, 3)"


In [47]:
first_last_seen.to_csv("first_last_seen.csv")

In [64]:
min_ver_per_dev_type = (
    df.loc[df["os_version_key"].notna(), ["device_type", "os_version_key", "os_version"]]
        .sort_values(["device_type", "os_version_key"], kind="mergesort")
        .groupby("device_type", as_index=False)
        .first()
        .rename(
            columns={
                "os_version_key": "min_os_version_key",
                "os_version": "min_os_version",
            }
        )
)
min_ver_per_dev_type

,device_type,min_os_version_key,min_os_version
0,Mac,"(11, 0, 1)",11.0.1
1,VisionPro,"(1, 0, 2)",1.0.2
2,iPad,"(11, 0, 1)",11.0.1
3,iPhone,"(12, 0)",12.0


In [95]:
first_last_seen2 = (
    first_last_seen.merge(min_ver_per_dev_type, on="device_type", how="left")
)

first_last_seen2["first_os_version"] = first_last_seen2["first_os_version"].where(
    first_last_seen2["first_os_version_key"] != first_last_seen2["min_os_version_key"],
    "\\textcolor{gray}{" + first_last_seen2["min_os_version"].astype(str) + "}"
)
first_last_seen2

,device_type,name,first_os_version,first_os_version_key,last_os_version,last_os_version_key,min_os_version_key,min_os_version
0,Mac,device-key-signature,13.0,"(13, 0)",26.3,"(26, 3)","(11, 0, 1)",11.0.1
1,Mac,ec-version,\textcolor{gray}{11.0.1},"(11, 0, 1)",26.3,"(26, 3)","(11, 0, 1)",11.0.1
2,Mac,is-c2k-equipment,\textcolor{gray}{11.0.1},"(11, 0, 1)",26.3,"(26, 3)","(11, 0, 1)",11.0.1
3,Mac,is-green-tea,12.0.1,"(12, 0, 1)",26.3,"(26, 3)","(11, 0, 1)",11.0.1
4,Mac,is-web-client,13.0,"(13, 0)",26.3,"(26, 3)","(11, 0, 1)",11.0.1
...,...,...,...,...,...,...,...,...
477,iPhone,supports-update-attachments-v1,\textcolor{gray}{12.0},"(12, 0)",26.3,"(26, 3)","(12, 0)",12.0
478,iPhone,supports-uriless-membership-updates,26.0,"(26, 0)",26.3,"(26, 3)","(12, 0)",12.0
479,iPhone,supports-user-driven-call-activation,18.4,"(18, 4)",26.3,"(26, 3)","(12, 0)",12.0
480,iPhone,supports-uwb,15.0,"(15, 0)",26.3,"(26, 3)","(12, 0)",12.0


In [96]:
first_table = first_last_seen2.pivot(columns="device_type", index="name")[["first_os_version"]].reset_index()
first_table

name          first_os_version  \
device_type                                                             Mac   
0                            device-key-signature                      13.0   
1                                      ec-version  \textcolor{gray}{11.0.1}   
2                                is-c2k-equipment  \textcolor{gray}{11.0.1}   
3                                    is-green-tea                    12.0.1   
4                                   is-web-client                      13.0   
..                                            ...                       ...   
118                supports-update-attachments-v1  \textcolor{gray}{11.0.1}   
119           supports-uriless-membership-updates                      26.0   
120          supports-user-driven-call-activation                      15.4   
121                                  supports-uwb                    12.0.1   
122                              supports-zelkova                      14.0   

                                                                \
device_type                VisionPro                      iPad   
0            \textcolor{gray}{1.0.2}                      16.1   
1            \textcolor{gray}{1.0.2}                      13.1   
2            \textcolor{gray}{1.0.2}  \textcolor{gray}{11.0.1}   
3            \textcolor{gray}{1.0.2}                      15.0   
4            \textcolor{gray}{1.0.2}                      16.1   
..                               ...                       ...   
118          \textcolor{gray}{1.0.2}  \textcolor{gray}{11.0.1}   
119                             26.0                      26.0   
120                             26.0                      18.4   
121          \textcolor{gray}{1.0.2}                      15.0   
122          \textcolor{gray}{1.0.2}                      17.0   

                                     
device_type                  iPhone  
0                              15.4  
1                              13.0  
2            \textcolor{gray}{12.0}  
3                              15.0  
4                              15.4  
..                              ...  
118          \textcolor{gray}{12.0}  
119                            26.0  
120                            18.4  
121                            15.0  
122                            17.0  

[123 rows x 5 columns]

In [97]:
print("Name & macOS & visionOS & iPadOS & iOS \\\\")
def tex(v):
    return "--" if pd.isna(v) else v
    
for _, frow in first_table.iterrows():
    print(f"\\texttt{{{frow.iloc[0]}}}", tex(frow.iloc[1]), tex(frow.iloc[2]), tex(frow.iloc[3]), tex(frow.iloc[4]), sep=" & ", end="")
    print(" \\\\")

Name & macOS & visionOS & iPadOS & iOS \\
\texttt{device-key-signature} & 13.0 & \textcolor{gray}{1.0.2} & 16.1 & 15.4 \\
\texttt{ec-version} & \textcolor{gray}{11.0.1} & \textcolor{gray}{1.0.2} & 13.1 & 13.0 \\
\texttt{is-c2k-equipment} & \textcolor{gray}{11.0.1} & \textcolor{gray}{1.0.2} & \textcolor{gray}{11.0.1} & \textcolor{gray}{12.0} \\
\texttt{is-green-tea} & 12.0.1 & \textcolor{gray}{1.0.2} & 15.0 & 15.0 \\
\texttt{is-web-client} & 13.0 & \textcolor{gray}{1.0.2} & 16.1 & 15.4 \\
\texttt{kt-version} & \textcolor{gray}{11.0.1} & \textcolor{gray}{1.0.2} & 13.1 & 13.0 \\
\texttt{nicknames-version} & \textcolor{gray}{11.0.1} & \textcolor{gray}{1.0.2} & 13.1 & 13.0 \\
\texttt{optionally-receive-typing-indicators} & \textcolor{gray}{11.0.1} & \textcolor{gray}{1.0.2} & \textcolor{gray}{11.0.1} & \textcolor{gray}{12.0} \\
\texttt{prefers-sdr} & \textcolor{gray}{11.0.1} & \textcolor{gray}{1.0.2} & 14.0 & 14.0 \\
\texttt{sender-key-message-version} & 15.0 & 2.0 & 18.0 & 18.0 \\
\texttt{s

### Aggregated Device Type Cap Map

In [48]:
(
    df
        .assign(_has_value=df["typed_value"].notna())
        .sort_values(["device_type", "os_version_key", "name", "_has_value"], ascending=[True, True, True, False])
        .groupby(["device_type", "os_version_key", "name"])
        .first()
        .reset_index()
        [["device_type", "os_version_key", "name", "typed_value"]]
        .to_csv("all_caps_version.csv", index=False)
)

### Device Cap Map

In [49]:
(
    df
        [["device_type", "device_model", "os_version", "name", "typed_value"]]
        .to_csv("all_caps_devices.csv", index=False)
)